# 18 Differential Abundance and Sex

We use ANCOM-BC2 to test whether fungal genus abundance is associated with sex (`sex_sample`). We use the unrarefied genus count table from notebook 14 and keep the same samples for the unadjusted and adjusted models. The adjusted model accounts for age and settlement type, as in notebook 11.

## 1. Setup

### 1.1 Imports

In [1]:
import pandas as pd
import qiime2 as q2
from qiime2 import Artifact, Metadata

In [2]:
metadata = "../data/raw/Final_metadata_with_GHSL_exclude_other_sex.tsv"
in_dir = "../data/processed_ignored/14_differential_abundance_setup"
out_dir = "../data/processed_ignored/18_differential_abundance_sex"
vis_dir = "../data/processed/18_differential_abundance_sex"

unadjusted_dir = f"{vis_dir}/unadjusted_ancombc2" 
adjusted_dir = f"{vis_dir}/adjusted_ancombc2"

genus_table = f"{in_dir}/table_fungi_l6.qza"
sex_table = f"{out_dir}/table_fungi_l6_sex.qza"

sex_unadjusted_qza = f"{out_dir}/ancombc2_sex_unadjusted.qza"
sex_adjusted_qza = f"{out_dir}/ancombc2_sex_adjusted.qza"

sex_unadjusted_tabulate = f"{unadjusted_dir}/tabulate_unadjusted_sex.qzv"
sex_adjusted_tabulate = f"{adjusted_dir}/tabulate_adjusted_sex.qzv"

sex_unadjusted_barplot = f"{unadjusted_dir}/barplot_unadjusted_sex.qzv"
sex_adjusted_barplot = f"{adjusted_dir}/barplot_adjusted_sex.qzv"

!mkdir -p "$out_dir" "$vis_dir"
!mkdir -p "$unadjusted_dir" "$adjusted_dir"

### 1.3 Load inputs

We load the genus count table from notebook 14 and the cleaned sample metadata used in notebook 09.

In [3]:
genus_artifact = Artifact.load(genus_table)
sample_metadata = Metadata.load(metadata)

print("Table type:", genus_artifact.type)
print("Samples in metadata:", sample_metadata.id_count)

Table type: FeatureTable[Frequency]
Samples in metadata: 149


## 2. Select samples

### 2.1 Keep complete cases

We keep samples with settlement type, age, and sex information so both models use the same samples.

In [4]:
!qiime feature-table filter-samples \
    --i-table "$genus_table" \
    --m-metadata-file "$metadata" \
    --p-where "[Settlement_Type] IS NOT NULL AND [age_years_sample] IS NOT NULL AND [sex_sample] IS NOT NULL" \
    --o-filtered-table "$sex_table"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/18_differential_abundance_sex/table_fungi_l6_sex.qza


### 2.2 Summarize selected samples

We count the samples in each sex group after filtering for complete settlement, age, and sex information.

In [5]:
sex_counts = Artifact.load(sex_table).view(pd.DataFrame)
selected_metadata = sample_metadata.to_dataframe().loc[sex_counts.index]

print("Selected samples:", len(selected_metadata))
print(selected_metadata["sex_sample"].value_counts())

Selected samples: 141
sex_sample
male      74
female    67
Name: count, dtype: int64


## 3. Run ANCOM-BC2

We use ANCOM-BC2 because some fungal genera were detected in only a few samples. When a genus has many zero counts, a result can depend on the small value used in place of zero. ANCOM-BC2 checks this by trying several small values and seeing whether the result stays significant. 

We add `--p-diff-robust` to both models to see which results stay significant when ANCOM-BC2 tries different small values for the zero counts.

### 3.1 Unadjusted ANCOM-BC2

We test whether fungal genus abundance differs by sex. We use male as the reference group and compare female samples with male samples.

In [6]:
!qiime composition ancombc2 \
    --i-table "$sex_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "sex_sample" \
    --p-reference-levels "sex_sample::male" \
    --p-diff-robust \
    --o-ancombc2-output "$sex_unadjusted_qza"

Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/18_differential_abundance_sex/ancombc2_sex_unadjusted.qza


In [7]:
!qiime composition tabulate \
    --i-data "$sex_unadjusted_qza" \
    --o-visualization "$sex_unadjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$sex_unadjusted_qza" \
    --o-visualization "$sex_unadjusted_barplot"


Saved Visualization to: ../data/processed/18_differential_abundance_sex/unadjusted_ancombc2/tabulate_unadjusted_sex.qzv
Saved Visualization to: ../data/processed/18_differential_abundance_sex/unadjusted_ancombc2/barplot_unadjusted_sex.qzv


In [8]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(sex_unadjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

for group in ["female"]:
    contrast = f"sex_sample::{group}"

    results = pd.DataFrame({
        "log_fold_change": tables["lfc"][contrast],
        "q_value": tables["q"][contrast],
        "diff_robust": tables["diff_robust"][contrast],
    })

    significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")
    print(f"{group} vs. male — {len(significant)} taxa with q < 0.05")
    display(significant)

female vs. male — 3 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Saccharomyces,2.163698,0.000124,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;__,-1.934682,0.002720,False
k__Fungi;p__Mucoromycota;c__Mucoromycetes;o__Mucorales;f__Mucoraceae;g__Mucor,1.769473,0.003296,False


#### Unadjusted results

| Taxon | Log fold change | q-value | diff_robust |
|---|---:|---:|:---:|
| *Saccharomyces* | +2.164 | 0.000124 | False |
| Dipodascaceae* | −1.935 | 0.002720 | False |
| *Mucor* | +1.769 | 0.003296 | False |

*genus unidentified

Three taxa had q < 0.05. Compared with male participants, female participants had higher estimated abundance of *Saccharomyces* and *Mucor* and lower estimated abundance of the unidentified Dipodascaceae taxon. None of the three results passed the sensitivity check for zero counts.

### 3.2 Adjusted ANCOM-BC2

We compare Omnivore diet samples with other diet labels after accounting for age and settlement. We use the same 139 samples as in the unadjusted model.

In [9]:
!qiime composition ancombc2 \
    --i-table "$sex_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "age_years_sample + Settlement_Type + sex_sample" \
    --p-reference-levels "sex_sample::male" \
    --p-diff-robust \
    --o-ancombc2-output "$sex_adjusted_qza"

!qiime composition tabulate \
    --i-data "$sex_adjusted_qza" \
    --o-visualization "$sex_adjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$sex_adjusted_qza" \
    --o-visualization "$sex_adjusted_barplot"


Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/18_differential_abundance_sex/ancombc2_sex_adjusted.qza
Saved Visualization to: ../data/processed/18_differential_abundance_sex/adjusted_ancombc2/tabulate_adjusted_sex.qzv
Saved Visualization to: ../data/processed/18_differential_abundance_sex/adjusted_ancombc2/barplot_adjusted_sex.qzv


In [10]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(sex_adjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

for group in ["female"]:
    contrast = f"sex_sample::{group}"

    results = pd.DataFrame({
        "log_fold_change": tables["lfc"][contrast],
        "q_value": tables["q"][contrast],
        "diff_robust": tables["diff_robust"][contrast],
    })

    significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")
    print(f"{group} vs. male — {len(significant)} taxa with q < 0.05")
    display(significant)

female vs. male — 6 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Saccharomyces,3.051837,0.000000e+00,False
k__Fungi;p__Mucoromycota;c__Mucoromycetes;o__Mucorales;f__Mucoraceae;g__Mucor,2.441321,3.504000e-07,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Debaryomycetaceae;g__Debaryomyces,1.614680,2.795790e-05,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Oligophagozyma,1.644239,2.527675e-04,False
k__Fungi;p__Basidiomycota;c__Tremellomycetes;o__Filobasidiales;f__Filobasidiaceae;g__Naganishia,1.574359,3.338421e-04,False
k__Fungi;p__Ascomycota;c__Dothideomycetes;o__Cladosporiales;f__Cladosporiaceae;g__Cladosporium,1.012463,5.980163e-03,False


#### Adjusted results

**Female vs. male**

| Taxon | Log fold change | q-value | diff_robust |
|---|---:|---:|:---:|
| *Saccharomyces* | +3.052 | 0 | False |
| *Mucor* | +2.441 | 3.50 × 10⁻⁷ | False |
| *Debaryomyces* | +1.615 | 2.80 × 10⁻⁵ | False |
| *Oligophagozyma* | +1.644 | 0.000253 | False |
| *Naganishia* | +1.574 | 0.000334 | False |
| *Cladosporium* | +1.012 | 0.005980 | False |

After adjustment, all six taxa had higher estimated abundance in female participants than in male participants (q < 0.05). *Saccharomyces* and *Mucor* showed this pattern in both models. None of the six adjusted results passed the sensitivity check for zero counts.

## 4. Discussion

*Saccharomyces* and *Mucor* had higher estimated abundance in female than in male participants in both models. Four other genera reached q < 0.05 only after adjustment. None of the findings passed the sensitivity check for zero counts, so the evidence for sex-related differences remains limited.